# 06 — Goal 2: participant persistence

Visualizes participant-level variance persistence without calling it test–retest reliability.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Main output: participant persistence ICC estimates and model-status table. Review sparse, under-supported, or nonconverged features; never relabel this as reliability.

In [ ]:
STAGE, FIGURES, TABLES = stage_directories(Path("04_analysis") / "goal2")
persistence = read_table(OUTPUT / "04_analysis" / "descriptive" / "participant_persistence_not_reliability")
save_table(persistence, TABLES, "participant_persistence")
display(persistence)

plot = persistence.loc[persistence["status"].eq("ok")].sort_values("persistence_icc")
fig, ax = plt.subplots(figsize=(10, max(5, 0.28 * len(plot))))
sns.barplot(data=plot, x="persistence_icc", y="feature", color="#59A14F", ax=ax)
ax.set(xlim=(0, 1), title="Participant rank persistence (not test–retest reliability)", xlabel="Variance-partition ICC", ylabel="")
fig.tight_layout()
save_figure(fig, FIGURES, "participant_persistence")
plt.show()

status = persistence["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="features")
save_table(status, TABLES, "persistence_model_status")
goal2_ready = stage_gate(
    "Goal 2",
    status.loc[status["status"].astype(str).str.startswith("model_failed"), "features"].sum() == 0,
    status.loc[~status["status"].eq("ok")].astype(str).agg(": ".join, axis=1).tolist(),
    "Proceed to Goal 3 after documenting every skipped/under-supported feature.",
)